# NREL quickstart — NSRDB solar + WIND Toolkit (live)

The `nrel` backend fetches **solar and wind resource time series** from the
US NREL/NLR [Developer Network](https://developer.nlr.gov/) keyed CSV download
API. Unlike the keyless backends it needs a **free API key + the email that
registered it** (`api_key=` / `email=`, or the `NREL_API_KEY` / `NREL_EMAIL`
environment variables) — register at <https://developer.nlr.gov/signup/>.

This notebook pulls one year of **hourly** NSRDB solar data and one year of
**WIND Toolkit** wind data for a single point (Denver, Colorado) and inspects
the returned `pandas.DataFrame`s.

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt

from earthlens.earthlens import EarthLens

# Credentials come from the environment; they are never printed.
API_KEY = os.environ["NREL_API_KEY"]
EMAIL = os.environ["NREL_EMAIL"]

OUT = tempfile.mkdtemp()  # downloads also write a CSV here

## Download one year of NSRDB hourly solar

`data_source="nrel"` uses the default `nsrdb-psm3` product (NSRDB GOES
Aggregated PSM v4). `variables=` lists the attributes; a single
`point=(lat, lon)` over one year is one keyed CSV call. `download()` returns
the hourly table and writes it to `path` as CSV.

In [ ]:
solar = EarthLens(
    data_source="nrel",
    variables=["ghi", "dni", "dhi", "air_temperature"],
    start="2020-01-01",
    end="2020-12-31",
    point=(39.74, -105.18),  # Denver, Colorado
    api_key=API_KEY,
    email=EMAIL,
    path=OUT,
).download(progress_bar=False)

print(solar.shape)
solar.head()

Each row is one hour. `GHI` / `DNI` / `DHI` are the global / direct / diffuse
irradiance components (W/m²), `Temperature` the air temperature (°C); the
`lat` / `lon` / `year` / `product` columns tag the sampled point. The `time`
column is assembled from the CSV's Year/Month/Day/Hour/Minute fields.

In [ ]:
solar.dtypes

## Plot a few days of irradiance

A short summer window shows the diurnal cycle of `GHI`.

In [ ]:
window = solar[(solar["time"] >= "2020-06-01") & (solar["time"] < "2020-06-04")]

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(window["time"], window["GHI"])
ax.set_ylabel("GHI  [W/m²]")
ax.set_title("NSRDB GHI — 1–3 June 2020 (Denver, 39.74°N, -105.18°E)")
fig.autofmt_xdate()
plt.show()

## Download WIND Toolkit hourly wind

The `"wind-toolkit"` alias selects the WTK product (`product="wtk"`). WTK
attributes are per hub height (`windspeed_100m`, `winddirection_100m`, …); the
CONUS archive covers roughly 2007–2014.

In [ ]:
wind = EarthLens(
    data_source="wind-toolkit",
    variables=["windspeed_100m", "winddirection_100m", "temperature_100m"],
    start="2012-01-01",
    end="2012-12-31",
    point=(39.74, -105.18),
    api_key=API_KEY,
    email=EMAIL,
    path=OUT,
).download(progress_bar=False)

print(wind.shape)
wind.head()

## Inspect the product catalog

The bundled catalog maps each product id to its source, endpoint, default
attributes, and columns. Select a product with `product=` or the
`nsrdb` / `wind-toolkit` facade aliases.

In [ ]:
from earthlens.nrel import Catalog

catalog = Catalog()
print(catalog.available())
catalog.get("nsrdb-psm3")

## Takeaway

One keyed call per `(point, year)` returns a tidy hourly `DataFrame` you can
resample or plot directly. NREL already returns the resolved hourly / TMY
series, so the facade rejects `aggregate=`; use pandas
(`df.set_index("time").resample("1D").mean()`) for a coarser cadence. The CSV
API is rate-limited (≤1 req/s, 5000/day), so a bbox × multi-year request fans
out into many throttled calls — coarsen `spacing_deg` or narrow the years.